In [3]:
import os
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from mlxtend.frequent_patterns import association_rules, fpgrowth
from mlxtend.preprocessing import TransactionEncoder
from sqlalchemy import bindparam, create_engine, text
from sqlalchemy.engine import URL


In [4]:
TRAINING_MONTHS = 3
HOLDOUT_MONTHS = 1
EXPERIMENT_END_DATE = "2026-07-31"  # Explicit YYYY-MM-DD; None selects a conservative completed month.

# Optional store subset for faster experiments, e.g. ("083", "001").
STORE_CODES = None

# Warehouse-read guardrails. 130 days accommodates the default 3+1 calendar-month experiment.
MAX_EXPERIMENT_DAYS = 130
SQL_CHUNK_SIZE = 250_000

# Provisional product exclusions until a governed SKU-eligibility contract exists
EXCLUDED_NAME_PATTERNS = ("pepito bag", )

# Mining guardrails
MIN_SUPPORT_COUNT = 5
MIN_LIFT = 1.05
MAX_ITEMSET_LEN = 3
MIN_STORE_TRAIN_BASKETS = 100

# Offline evaluation
TOP_K = 5
EVAL_MAX_BASKETS_PER_STORE = 2_000
RANDOM_SEED = 42

In [5]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

env_path = project_root / ".env"
if env_path.exists():
    load_dotenv(env_path)
else:
    load_dotenv(project_root / ".env.development")

source_config = {
    "host": (os.getenv("SOURCE_MSSQL_HOST") or "").strip(),
    "port": int(os.getenv("SOURCE_MSSQL_PORT") or 1433),
    "database": (os.getenv("SOURCE_MSSQL_DATABASE") or "").strip(),
    "user": (os.getenv("SOURCE_MSSQL_USER") or "").strip(),
    "password": (os.getenv("SOURCE_MSSQL_PASSWORD") or "").strip(),
    "driver": (os.getenv("SOURCE_MSSQL_DRIVER") or "ODBC Driver 18 for SQL Server").strip(),
}

missing_source = [
    key for key in ("host", "port", "database", "user", "password", "driver")
    if not source_config.get(key)
]

if missing_source:
    raise ValueError(f"Missing SQL Server configuration values: {', '.join(missing_source)}")

try:
    source_port = int(source_config["port"])
except ValueError as exc:
    raise ValueError("SOURCE_MSSQL_PORT must be an integer") from exc

MIN_SUPPORT_RATIO = float(os.getenv("MIN_SUPPORT_RATIO") or 0.0001)
MIN_CONFIDENCE = float(os.getenv("MIN_CONFIDENCE") or 0.01)

print(
    "SQL Server source:",
    f"{source_config['host']}:{source_config['port']}/{source_config['database']}"
)
print("Driver: ", source_config["driver"])
print("Minimum support ratio: ", MIN_SUPPORT_RATIO)
print("Minimum confidence: ", MIN_CONFIDENCE)

SQL Server source: 192.168.85.55:1433/DBWH_8555
Driver:  ODBC Driver 18 for SQL Server
Minimum support ratio:  0.001
Minimum confidence:  0.0


In [6]:
connection_url = URL.create(
    "mssql+pyodbc",
    username=source_config["user"],
    password=source_config["password"],
    host=source_config["host"],
    port=source_port,
    database=source_config["database"],
    query={
        "driver": source_config["driver"],
        "Encrypt": "yes",
        "TrustServerCertificate": "yes",
    }
)

source_engine = create_engine(
    connection_url,
    pool_pre_ping=True,
    future=True,
)

with source_engine.connect() as conn:
    connection_test = conn.execute(text("SELECT 1")).scalar()

print("SQL Server connection OK: ", connection_test)

SQL Server connection OK:  1


In [7]:
if EXPERIMENT_END_DATE:
    source_max_date = None
    holdout_end = pd.Timestamp(EXPERIMENT_END_DATE).normalize()
else:
    latest_date_query = text(
        """
        SELECT
            dd.[date] AS max_trx_date,
            latest.[DateKey] AS max_date_key
        FROM
        (
            SELECT TOP (1)
                fstn.[DateKey]
            FROM [dbo].[FactSalesTrxNew] AS fstn
            ORDER BY fstn.[DateKey] DESC
        ) AS latest
        INNER JOIN [dbo].[DimDate] AS dd
            ON dd.[DateKey] = latest.[DateKey]
        """
    )

    with source_engine.connect() as conn:
        latest_date_df = pd.read_sql(latest_date_query, conn)

    if latest_date_df.empty:
        raise RuntimeError("FactSalesTrxNew returned no transaction date")

    source_max_date = pd.Timestamp(latest_date_df["max_trx_date"].iloc[0]).normalize()
    holdout_end = source_max_date.replace(day=1) - pd.Timedelta(days=1)



holdout_start = (holdout_end - pd.DateOffset(months=HOLDOUT_MONTHS - 1)).replace(day=1)
train_end = holdout_start - pd.Timedelta(days=1)
train_start = (train_end - pd.DateOffset(months=TRAINING_MONTHS - 1)).replace(day=1)

if train_start >= train_end or holdout_start > holdout_end:
    raise ValueError("Invalid experiment window; review TRAINING_MONTHS, HOLDOUT_MONTHS, and EXPERIMENT_END_DATE")

experiment_days = (holdout_end - train_start).days + 1
if experiment_days > MAX_EXPERIMENT_DAYS:
    raise ValueError(
        f"Experiment request {experiment_days:,} days from FactSalesTrxNew; "
        f"the notebook guardrail is {MAX_EXPERIMENT_DAYS:,} days. Consider "
        f"reducing TRAINING_MONTHS and/or HOLDOUT_MONTHS"
    )

# Resolve the calendar window to the clustered fact-table key before the large read.
date_dim_query = text(
    """
    SELECT 
        dd.[datekey] AS DateKey,
        dd.[date] AS TRX_date
    FROM [dbo].[DimDate] AS dd
    WHERE dd.[date] BETWEEN :start_date AND :end_date
    ORDER BY dd.[datekey]
    """
)

with source_engine.connect() as conn:
    date_dim_df = pd.read_sql_query(
        date_dim_query,
        conn,
        params={
            "start_date": train_start.date(),
            "end_date": holdout_end.date()
        }
    )

if date_dim_df.empty:
    raise RuntimeError("DimDate returned no rows for the experiment window")

if date_dim_df["DateKey"].duplicated().any():
    raise RuntimeError("DimDate.DateKey is not unique inside the experiment window")

start_date_key = int(date_dim_df["DateKey"].min())
end_date_key = int(date_dim_df["DateKey"].max())
date_dim_df["TRX_date"] = pd.to_datetime(date_dim_df["TRX_date"]).dt.normalize()

if source_max_date is not None:
    print("Source max transaction date: ", source_max_date.date())
else:
    print("Source max transaction date: None (using EXPERIMENT_END_DATE)")


if source_max_date is not None:
    print("Source max transaction date: ", source_max_date.date())
else:
    print("Source max transaction date: None (using EXPERIMENT_END_DATE)")

print("Training window: ", train_start.date(), "to", train_end.date())
print("Holdout window: ", holdout_start.date(), "to", holdout_end.date())
print("Fact DateKey range: ", start_date_key, "to", end_date_key)
print("Requested warehouse days: ", experiment_days)
print("Store filter: ", STORE_CODES if STORE_CODES else "None (ALL STORES)")

Source max transaction date: None (using EXPERIMENT_END_DATE)
Source max transaction date: None (using EXPERIMENT_END_DATE)
Training window:  2026-04-01 to 2026-06-30
Holdout window:  2026-07-01 to 2026-07-31
Fact DateKey range:  20260401 to 20260731
Requested warehouse days:  122
Store filter:  None (ALL STORES)


## Extract experiment transactions using the clustered DateKey range

In [8]:
fact_sql = """
    SELECT
        fstn.[DateKey],
        fstn.[StoreCode],
        fstn.[BillNo],
        fstn.[ItemCode],
        fstn.[POS_FINAL_QTY]
    FROM [dbo].[FactSalesTrxNew] AS fstn
    WHERE fstn.[DateKey] >= :start_date_key
        AND fstn.[DateKey] <= :end_date_key
        AND fstn.[POS_FINAL_QTY] > 0
        AND fstn.[StoreCode] IS NOT NULL
        AND fstn.[BillNo] IS NOT NULL
        AND fstn.[ItemCode] IS NOT NULL
    """


fact_params = {
    "start_date_key": start_date_key,
    "end_date_key": end_date_key
}

if STORE_CODES:
    extract_query = text(fact_sql + "\n AND fstn.[StoreCode] IN :store_codes").bindparams(
        bindparam("store_codes", expanding=True)
    )
    fact_params["store_codes"] = list(STORE_CODES)
else:
    extract_query = text(fact_sql)

item_dim_query = text(
    """
    SELECT
        di.[ITMCD] AS ItemCode,
        di.[ITEMLONGNAME]
    FROM [dbo].[DimItem] AS di
    WHERE di.[ITMCD] IS NOT NULL
    """
)

with source_engine.connect() as conn:
    item_dim_df = pd.read_sql_query(item_dim_query, conn)

item_dim_df["ItemCode"] = item_dim_df["ItemCode"].astype("string").str.strip()
if item_dim_df["ItemCode"].duplicated().any():
    duplicate_count = int(item_dim_df["ItemCode"].duplicated(keep=False).sum())
    raise RuntimeError(
        f"DimItem contains {duplicate_count:,} rows with duplicated ITMCD values; "
        "resolve the item-dimension grain before merging it into the experiment."
    )

fact_chunks = []
extracted_rows = 0

with source_engine.connect() as conn:
    chunk_iterator = pd.read_sql_query(
        extract_query,
        conn,
        params=fact_params,
        chunksize=SQL_CHUNK_SIZE
    )
    for chunk_number, chunk in enumerate(chunk_iterator, start=1):
        extracted_rows += len(chunk)
        fact_chunks.append(chunk)
        print(
            f"Fetched chunk {chunk_number:,} with {len(chunk):,} rows; total extracted: {extracted_rows:,}"
        )

if not fact_chunks:
    raise RuntimeError("FactSalesTrxNew returned no rows for the experiment window")

fact_df = pd.concat(fact_chunks, ignore_index=True)
fact_df["ItemCode"] = fact_df["ItemCode"].astype("string").str.strip()

raw_df = fact_df.merge(
    item_dim_df,
    how="left",
    on="ItemCode",
    validate="many_to_one"
).merge(
    date_dim_df,
    how="left",
    on="DateKey",
    validate="many_to_one"
)

print(f"Extracted fact rows: {len(fact_df):,}")
print(f"Stores: {fact_df['StoreCode'].nunique():,}")
print(f"Enriched rows: {len(raw_df):,} (after merging DimItem and DimDate)")
raw_df.head()

Fetched chunk 1 with 250,000 rows; total extracted: 250,000
Fetched chunk 2 with 250,000 rows; total extracted: 500,000
Fetched chunk 3 with 250,000 rows; total extracted: 750,000
Fetched chunk 4 with 250,000 rows; total extracted: 1,000,000
Fetched chunk 5 with 250,000 rows; total extracted: 1,250,000
Fetched chunk 6 with 250,000 rows; total extracted: 1,500,000
Fetched chunk 7 with 250,000 rows; total extracted: 1,750,000
Fetched chunk 8 with 250,000 rows; total extracted: 2,000,000
Fetched chunk 9 with 250,000 rows; total extracted: 2,250,000
Fetched chunk 10 with 250,000 rows; total extracted: 2,500,000
Fetched chunk 11 with 250,000 rows; total extracted: 2,750,000
Fetched chunk 12 with 250,000 rows; total extracted: 3,000,000
Fetched chunk 13 with 250,000 rows; total extracted: 3,250,000
Fetched chunk 14 with 250,000 rows; total extracted: 3,500,000
Fetched chunk 15 with 250,000 rows; total extracted: 3,750,000
Fetched chunk 16 with 250,000 rows; total extracted: 4,000,000
Fetched

,DateKey,StoreCode,BillNo,ItemCode,POS_FINAL_QTY,ITEMLONGNAME,TRX_date
0,20260401,002,8C20020017003,101022011265,0.8480,DORY FILLET KG,2026-04-01
1,20260401,002,8C20020017003,101024011692,1.1778,SEMANGKA MERAH NB,2026-04-01
2,20260401,002,8C20020017004,101003055012,1.0000,NORIGO GIANT SHEET ORIGINAL 3.2GR,2026-04-01
3,20260401,002,8C20020017004,101023011588,0.1820,BROKOLI LOKAL KG,2026-04-01
4,20260401,002,8C20020017004,101023031281,0.1939,RUMPUT LAUT PUTIH KG,2026-04-01


## Validate source schema and transaction grain

In [9]:
required_columns = {
    "DateKey",
    "TRX_date",
    "StoreCode",
    "BillNo",
    "ItemCode",
    "ITEMLONGNAME",
    "POS_FINAL_QTY"
}

missing_columns = sorted(required_columns.difference(raw_df.columns))
if missing_columns:
    raise ValueError(f"Missing required source columns: {missing_columns}")

raw_df["TRX_date"] = pd.to_datetime(raw_df["TRX_date"]).dt.normalize()

core_nulls = raw_df[["DateKey", "TRX_date", "StoreCode", "BillNo", "ItemCode", "POS_FINAL_QTY"]].isna().sum()
print("Core null counts:")
display(core_nulls.to_frame("null_count"))

unmapped_dates = int(raw_df["TRX_date"].isna().sum())
if unmapped_dates:
    raise RuntimeError(f"{unmapped_dates:,} fact rows did not map to DimDate; check the fact-to-dimension join and the experiment window")

observed_min_key = int(raw_df["DateKey"].min())
observed_max_key = int(raw_df["DateKey"].max())
if observed_min_key < start_date_key or observed_max_key > end_date_key:
    raise RuntimeError(
        f"FactSalesTrxNew.DateKey range {observed_min_key:,} to {observed_max_key:,} "
        f"exceeds the requested experiment range {start_date_key:,} to {end_date_key:,}"
    )

bill_date_counts = (
    raw_df.dropna(subset=["StoreCode", "BillNo", "TRX_date"]).groupby(["StoreCode", "BillNo"])["TRX_date"].nunique()
)

reused_bill_keys = bill_date_counts[bill_date_counts > 1]

print(f"Observed DateKey range: {observed_min_key} to {observed_max_key}")
print(f"Store/BillNo values appearing on multiple dates: {len(reused_bill_keys):,} (should be 0 for a clean fact table)")
if not reused_bill_keys.empty:
    print("Sample reused Store/BillNo keys:")
    display(reused_bill_keys.sort_values(ascending=False).head(20).to_frame("distinct_dates"))

Core null counts:


,null_count
DateKey,0
TRX_date,0
StoreCode,0
BillNo,0
ItemCode,0
POS_FINAL_QTY,0


Observed DateKey range: 20260401 to 20260731
Store/BillNo values appearing on multiple dates: 0 (should be 0 for a clean fact table)


## Clean, normalize, and deduplicate basket lines

In [10]:
clean_df = raw_df.copy()
cleaning_steps = []

def apply_filter(frame, step_name, keep_mask):
    before = len(frame)
    result = frame.loc[keep_mask].copy()
    cleaning_steps.append(
        {
            "step": step_name,
            "rows_before": before,
            "rows_after": len(result),
            "rows_removed": before - len(result)
        }
    )
    return result

clean_df = apply_filter(
    clean_df,
    "required basked fields are non-null",
    clean_df[["TRX_date", "StoreCode", "BillNo", "ItemCode"]].notna().all(axis=1)
)

clean_df["POS_FINAL_QTY"] = pd.to_numeric(clean_df["POS_FINAL_QTY"], errors="coerce")
clean_df = apply_filter(
    clean_df,
    "POS_FINAL_QTY > 0",
    clean_df["POS_FINAL_QTY"].gt(0)
)

if EXCLUDED_NAME_PATTERNS:
    exclusion_regex = "|".join(EXCLUDED_NAME_PATTERNS)
    clean_df = apply_filter(
        clean_df,
        "product name is not provisionally excluded",
        ~clean_df["ITEMLONGNAME"].str.contains(exclusion_regex, case=False, regex=True)
    )

for column in ("StoreCode", "BillNo", "ItemCode"):
    clean_df[column] = clean_df[column].astype(str).str.strip()

basket_item_grain = ["StoreCode", "TRX_date", "BillNo", "ItemCode"]
before_dedup = len(clean_df)
clean_df = clean_df.drop_duplicates(subset=basket_item_grain).copy()
cleaning_steps.append(
    {
        "step": "deduplicate at basket-item grain",
        "rows_before": before_dedup,
        "rows_after": len(clean_df),
        "rows_removed": before_dedup - len(clean_df)
    }
)
display(pd.DataFrame(cleaning_steps))

,step,rows_before,rows_after,rows_removed
0,required basked fields are non-null,16022310,16022310,0
1,POS_FINAL_QTY > 0,16022310,16022310,0
2,product name is not provisionally excluded,16022310,15496496,525814
3,deduplicate at basket-item grain,15496496,15468782,27714


## Build canonical store baskets

In [11]:
canonical_baskets = (
    clean_df.groupby(["StoreCode", "TRX_date", "BillNo"], as_index=False)["ItemCode"]
    .agg(lambda values: sorted(set(values)))
    .rename(columns={"ItemCode": "items"})
)

canonical_baskets["basket_id"] = (
    canonical_baskets["StoreCode"].astype(str)
    + "|"
    + canonical_baskets["TRX_date"].dt.strftime("%Y-%m-%d")
    + "|"
    + canonical_baskets["BillNo"].astype(str)
)
canonical_baskets["item_count"] = canonical_baskets["items"].map(len)

basket_summary = canonical_baskets["item_count"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
print(f"Canonical baskets: {len(canonical_baskets):,}")
print(f"Unique items: {clean_df['ItemCode'].nunique():,}")
display(basket_summary.to_frame("item_count"))

display(
    canonical_baskets.groupby("StoreCode")
    .agg(baskets=("basket_id", "size"), avg_items=("item_count", "mean"))
    .sort_values("baskets", ascending=False)
    .head(30)
)

Canonical baskets: 3,219,697
Unique items: 25,121


,item_count
count,3.219697e+06
mean,4.804422e+00
std,5.468131e+00
min,1.000000e+00
25%,1.000000e+00
50%,3.000000e+00
75%,6.000000e+00
90%,1.100000e+01
95%,1.500000e+01
99%,2.700000e+01


,baskets,avg_items
StoreCode,,
034,147960,4.240592
028,125723,5.390462
027,112646,5.766596
033,112559,4.871170
078,93982,5.603690
075,92161,6.237530
046,88029,5.736712
024,86524,6.273473
007,85321,4.010935


## Temporal Split

In [13]:
train_baskets = canonical_baskets[canonical_baskets["TRX_date"].between(train_start, train_end)].copy()
holdout_baskets = canonical_baskets[canonical_baskets["TRX_date"].between(holdout_start, holdout_end)].copy()

if train_baskets.empty:
    raise RuntimeError("Training basket set is empty"
                       )
if holdout_baskets.empty:
    raise RuntimeError("Holdout basket set is empty")

split_summary = pd.DataFrame(
    [
        # {
        #     "split": "training",
        #     "baskets": len(train_baskets),
        #     "unique_items": train_baskets["items"].explode().nunique(),
        #     "start_date": train_baskets["TRX_date"].min().date(),
        #     "end_date": train_baskets["TRX_date"].max().date()
        # },
        # {
        #     "split": "holdout",
        #     "baskets": len(holdout_baskets),
        #     "unique_items": holdout_baskets["items"].explode().nunique(),
        #     "start_date": holdout_baskets["TRX_date"].min().date(),
        #     "end_date": holdout_baskets["TRX_date"].max().date()
        # }
        {
            "split": "training",
            "start": train_start.date(),
            "end": train_end.date(),
            "baskets": len(train_baskets),
            "stores": train_baskets["StoreCode"].nunique(),
        },
        {
            "split": "holdout",
            "start": holdout_start.date(),
            "end": holdout_end.date(),
            "baskets": len(holdout_baskets),
            "stores": holdout_baskets["StoreCode"].nunique(),
        },
    ]
)
display(split_summary)

,split,start,end,baskets,stores
0,training,2026-04-01,2026-06-30,2369005,52
1,holdout,2026-07-01,2026-07-31,850692,52


## Association Mining

### Mine per-store frequent itemsets and association rules

In [ ]:
item_name_map = (
    clean_df.dropna(subset=["ItemCode"])
    .drop_duplicates(subset=["ItemCode"], keep="last")
    .set_index("ItemCode")[["ITEMLONGNAME"]]
    .fillna("")
    .to_dict()
)

def min_store_rules(store_code, store_baskets):
    transactions = [list(items) for items in store_baskets["items"]]
    n_baskets = len(transactions)

    if n_baskets < MIN_STORE_TRAIN_BASKETS:
        return pd.DataFrame(), {
            "store_code": store_code,
            "train_baskets": n_baskets,
            "effective_min_support": np.nan,
            "frequent_itemsets": 0,
            "rules": 0,
            "status": "skipped: insufficient training baskets"
        }

    effective_min_support = max(MIN_SUPPORT_RATIO, MIN_SUPPORT_COUNT / n_baskets)

    encoder = TransactionEncoder()
    encoded_sparse = encoder.fit(transactions).transform(transactions, sparse=True)
    basket_matrix = pd.DataFrame.sparse.from_spmatrix(
        encoded_sparse,
        columns=encoder.columns_,
    ).astype(pd.SparseDtype(bool, fill_value=False))

    frequent_itemsets = fpgrowth(
        basket_matrix,
        min_support=effective_min_support,
        use_colnames=True,
        max_len=MAX_ITEMSET_LEN
    )

    if frequent_itemsets.empty or not frequent_itemsets["itemsets"].map(lambda x: len(x) > 1).any():
        return pd.DataFrame(), {
            "store_code": store_code,
            "train_baskets": n_baskets,
            "effective_min_support": effective_min_support,
            "frequent_itemsets": len(frequent_itemsets),
            "rules": 0,
            "status": "no frequent itemsets found"
        }

    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=MIN_CONFIDENCE
    )

    if rules.empty:
        return rules, {
            "store_code": store_code,
            "train_baskets": n_baskets,
            "effective_min_support": effective_min_support,
            "frequent_itemsets": len(frequent_itemsets),
            "rules": 0,
            "status": "no rules found"
        }

    rules = rules.copy()
    rules["support_count"] = np.rint(rules["support"] * n_baskets).astype(int)
    rules = rules[
        rules["support_count"].ge(MIN_SUPPORT_COUNT)
        & rules["lift"].ge(MIN_LIFT)
        & rules["consequents"].map(lambda value: isinstance(value, frozenset) and len(value) == 1)
    ].copy()

    rules["store_code"] = store_code
    rules["train_baskets"] = n_baskets
    rules["antecedent_size"] = rules["antecedents"].map(len)
    rules["consequent_item"] = rules["consequents"].map(
        lambda value: next(iter(value)) if isinstance(value, frozenset) and len(value) == 1 else pd.NA
    )
    rules["consequent_name"] = rules["consequent_item"].map(item_name_map)
    rules["antecedent_items"] = rules["antecedents"].map(
        lambda value: ", ".join(sorted(str(item) for item in value))
        if isinstance(value, frozenset)
        else pd.NA
    )

    diagnostics = {
        "store_code": store_code,
        "train_baskets": n_baskets,
        "effective_min_support": effective_min_support,
        "frequent_itemsets": len(frequent_itemsets),
        "rules": len(rules),
        "status": "success" if not rules.empty else "rules removed by guardrails",
    }
    return rules, diagnostics

all_rule_frames = []
mining_diagnostics = []

for store_code, store_frame in train_baskets.groupby("StoreCode", sort=True):
    store_rules, diagnostics = min_store_rules(store_code, store_frame)
    mining_diagnostics.append(diagnostics)
    if not store_rules.empty:
        all_rule_frames.append(store_rules)

candidate_rules = (
    pd.concat(all_rule_frames, ignore_index=True)
    if all_rule_frames else pd.DataFrame()
)

mining_summary = pd.DataFrame(mining_diagnostics).sort_values("store_code").reset_index(drop=True)

print(f"Candidate rules after guardrails: {len(candidate_rules):,}")
display(mining_summary.sort_values(["rules", "train_baskets"], ascending=[False, False]).head(30))

Candidate rules after guardrails: 35,810


,store_code,train_baskets,effective_min_support,frequent_itemsets,rules,status
42,075,68529,0.001,2621,2970,success
23,038,52181,0.001,2445,2501,success
25,043,58273,0.001,1822,1658,success
15,027,84122,0.001,1851,1609,success
3,006,35885,0.001,1843,1519,success
8,014,54889,0.001,1652,1317,success
48,081,34553,0.001,1563,1228,success
17,029,43007,0.001,1571,1155,success
45,078,71326,0.001,1595,1094,success
13,024,63989,0.001,1690,1045,success


## Inspect Candidate Rules

In [19]:
if candidate_rules.empty:
    print("No candidate rules survived the current experimental thresholds.")
else:
    # # normalize older typoed columns from prior notebook runs
    # if "antecedent_items" not in candidate_rules.columns and "antecendent_items" in candidate_rules.columns:
    #     candidate_rules = candidate_rules.rename(columns={"antecendent_items": "antecedent_items"})
    # if "antecedent_size" not in candidate_rules.columns and "antecendent_size" in candidate_rules.columns:
    #     candidate_rules = candidate_rules.rename(columns={"antecendent_size": "antecedent_size"})

    rule_columns = [
        "store_code",
        "antecedent_items",
        "consequent_item",
        "consequent_name",
        "support_count",
        "support",
        "confidence",
        "lift",
        "leverage",
    ]
    display(
        candidate_rules.sort_values(
            ["lift", "confidence", "support_count"],
            ascending=False,
        )[rule_columns].head(50)
    )

,store_code,antecedent_items,consequent_item,consequent_name,support_count,support,confidence,lift,leverage
13883,032,101078059365,101078059369,NaN,19,0.001479,1.0,676.157895,0.001477
13884,032,101078059369,101078059365,NaN,19,0.001479,1.0,676.157895,0.001477
13885,032,101078059365,101078059366,NaN,19,0.001479,1.0,676.157895,0.001477
13886,032,101078059366,101078059365,NaN,19,0.001479,1.0,676.157895,0.001477
13887,032,101078059369,101078059366,NaN,19,0.001479,1.0,676.157895,0.001477
13888,032,101078059366,101078059369,NaN,19,0.001479,1.0,676.157895,0.001477
13889,032,"101078059365, 101078059369",101078059366,NaN,19,0.001479,1.0,676.157895,0.001477
13890,032,"101078059365, 101078059366",101078059369,NaN,19,0.001479,1.0,676.157895,0.001477
13891,032,"101078059366, 101078059369",101078059365,NaN,19,0.001479,1.0,676.157895,0.001477
13892,032,101078059367,101078059366,NaN,19,0.001479,1.0,676.157895,0.001477


## Offline Evaluation

### Recommendation Function

In [20]:
def recommend_cross_sell(observed_items, store_rules, top_k=TOP_K):
    if store_rules.empty:
        return []

    observed = set(observed_items)
    matched = store_rules[store_rules["antecedents"].map(lambda x: x.issubset(observed))].copy()

    if matched.empty:
        return []

    matched = matched[~matched["consequent_item"].isin(observed)].copy()
    if matched.empty:
        return []

    matched = matched.sort_values(
        ["confidence", "lift", "support_count", "leverage"],
        ascending=False
    ).drop_duplicates(subset=["consequent_item"], keep="first")
    return matched["consequent_item"].head(top_k).tolist()

def deterministic_hidden_item(basket_id, items, seed=RANDOM_SEED):
    ordered_items = sorted(items)
    digest = hashlib.sha256(f"{seed}|{basket_id}".encode("utf-8")).digest()
    index = int.from_bytes(digest[:8], byteorder="big") % len(ordered_items)
    return ordered_items[index]

### Evaluate Future Baskets

In [30]:
def stable_sample(frame, max_rows, seed):
    if len(frame) <= max_rows:
        return frame
    return frame.sample(n=max_rows, random_state=seed).sort_values("basket_id")

store_rule_groups = {
    store_code: frame.copy()
    for store_code, frame in candidate_rules.groupby("store_code")
} if not candidate_rules.empty else {}

evaluation_rows = []

for store_code, store_holdout in holdout_baskets.groupby("StoreCode", sort=True):
    eligible = store_holdout[store_holdout["item_count"].ge(2)].copy()
    eligible = stable_sample(eligible, EVAL_MAX_BASKETS_PER_STORE, RANDOM_SEED)
    store_rules = store_rule_groups.get(store_code, pd.DataFrame())

    for row in eligible.itertuples(index=False):
        target_item = deterministic_hidden_item(row.basket_id, row.items)
        observed_items = tuple(item for item in row.items if item != target_item)
        recommendations = recommend_cross_sell(observed_items, store_rules, top_k=TOP_K)
        recommendations_names = tuple(item_name_map.get("ITEMLONGNAME", {}).get(item, f"Unknown item ({item})") for item in recommendations)
        target_name = item_name_map.get("ITEMLONGNAME", {}).get(target_item, f"Unknown Item ({target_item})")
        hit = int(target_item in recommendations)

        evaluation_rows.append(
            {
                "store_code": store_code,
                "basket_id": row.basket_id,
                "business_date": row.TRX_date.date(),
                "observed_items_count": len(observed_items),
                "target_item": target_item,
                "target_item_name": item_name_map.get("ITEMLONGNAME", {}).get(target_item, ""),
                "target_name": target_name,
                "recommendations": recommendations,
                "recommendation_names": recommendations_names,
                "recommendation_count": len(recommendations),
                "hit_at_k": hit,
                # Exactly one relevant hidden item is used in this experiment.
                "precision_returned": hit / len(recommendations) if recommendations else 0.0,
                "precision_at_k": hit / TOP_K,
                # With one hidden relevant item, recall@K is equivalent
                # to whether that item appears in the Top-K list.
                "recall_at_k": float(hit / 1.0),
            }
        )

evaluation_df = pd.DataFrame(evaluation_rows)

evaluation_df["has_recommendation"] = (evaluation_df["recommendation_count"] > 0)
evaluation_df["full_top_k"] = (evaluation_df["recommendation_count"] >= TOP_K)

if evaluation_df.empty:
    raise RuntimeError("No holdout baskets were evaluable. Check the temporal window, store subset, or basket cleaning rules.")

evaluation_summary = pd.DataFrame(
    [
        {
            "K": TOP_K,
            "evaluated_baskets": len(evaluation_df),
            "hits": int(evaluation_df["hit_at_k"].sum()),
            f"HitRate@{TOP_K}": evaluation_df["hit_at_k"].mean(),
            f"Precision@{TOP_K}": evaluation_df["precision_at_k"].mean(),
            f"Recall@{TOP_K}": evaluation_df["recall_at_k"].mean(),
        }
    ]
)

coverage_summary = pd.DataFrame(
    [
        {
            "evaluated_baskets": len(evaluation_df),
            "baskets_with_recommendations": (
                evaluation_df["has_recommendation"].sum()
            ),
            "recommendation_coverage": (
                evaluation_df["has_recommendation"].mean()
            ),
            "full_top_k_baskets": (
                evaluation_df["full_top_k"].sum()
            ),
            "full_top_k_rate": (
                evaluation_df["full_top_k"].mean()
            ),
            "avg_recommendation_count": (
                evaluation_df["recommendation_count"].mean()
            ),
        }
    ]
)

pct_columns = [f"HitRate@{TOP_K}", f"Precision@{TOP_K}", f"Recall@{TOP_K}"]

display(
    evaluation_summary.copy().assign(
        **{
            col: evaluation_summary[col].map(lambda value: f"{value:.2%}")
            for col in pct_columns
        }
    )
)

,K,evaluated_baskets,hits,HitRate@5,Precision@5,Recall@5
0,5,104000,6305,6.06%,1.21%,6.06%


In [31]:
display(
    coverage_summary.copy().assign(
        **{
            col: coverage_summary[col].map(lambda value: f"{value:.2%}")
            for col in ["recommendation_coverage", "full_top_k_rate"]
        }
    )
)

,evaluated_baskets,baskets_with_recommendations,recommendation_coverage,full_top_k_baskets,full_top_k_rate,avg_recommendation_count
0,104000,59857,57.55%,36090,34.70%,2.191375


In [32]:
# Add a human-readable drill-down
evaluation_detail = evaluation_df[
    [
        "store_code",
        "basket_id",
        "business_date",
        "target_name",
        "recommendation_names",
        "hit_at_k",
        "precision_at_k",
        "recall_at_k",
    ]
].copy()

display(
    evaluation_detail
    .sort_values(
        ["hit_at_k", "store_code"],
        ascending=[False, True],
    )
    .head(20)
)

,store_code,basket_id,business_date,target_name,recommendation_names,hit_at_k,precision_at_k,recall_at_k
10,002,002|2026-07-01|8C20020035538,2026-07-01,MANGGIS,"(BINTANG RADLER CAN 320ML, MANGGIS, POP MIE AY...",1,0.2,1.0
21,002,002|2026-07-01|8C20020035598,2026-07-01,MERUBALI COCONUT CHIPS CHOCOLATE 50GR,"(MERUBALI COCONUT CHIPS CHOCOLATE 50GR, MANGGIS)",1,0.2,1.0
42,002,002|2026-07-01|8C40020029010,2026-07-01,BCC TIRAMISU BERLINER,"(BCC TIRAMISU BERLINER,)",1,0.2,1.0
71,002,002|2026-07-02|8C20020035654,2026-07-02,SUNPRIDE PISANG CAVENDISH,"(MERUBALI COCONUT CHIPS CHOCOLATE 50GR, FRUIT ...",1,0.2,1.0
79,002,002|2026-07-02|8C20020035720,2026-07-02,MERUBALI COCONUT CHIPS 50GR,"(MERUBALI COCONUT CHIPS 50GR, MANGGIS, TARI BA...",1,0.2,1.0
120,002,002|2026-07-02|8C40020029279,2026-07-02,BCC STRAWBERRY DONUT,"(BCC DONUT CHEESE, BCC STRAWBERRY DONUT, BCC D...",1,0.2,1.0
123,002,002|2026-07-02|8C40020029295,2026-07-02,BCC STRAWBERRY DONUT,"(BCC DONUT CHEESE, BCC STRAWBERRY DONUT, BCC D...",1,0.2,1.0
125,002,002|2026-07-02|8C40020029304,2026-07-02,RTE-P SALAD BUFFET PEPITO,"(RTE-P SALAD BUFFET PEPITO, BCC ALMOND CROISSA...",1,0.2,1.0
129,002,002|2026-07-02|8C50020025448,2026-07-02,BCC SOURDOUGH DINNER ROLL,"(BCC SOURDOUGH DINNER ROLL,)",1,0.2,1.0
152,002,002|2026-07-03|8C20020036021,2026-07-03,JERUK WOGAN,"(PEAR CENTURY, JERUK WOGAN, GRAPE RED GLOBE PR...",1,0.2,1.0


## Inspect misses and successful recommendations

In [33]:
print("Sample successful recommendations")
display(
    evaluation_df[evaluation_df["hit_at_k"].eq(1)]
    .head(20)[
        [
            "store_code",
            "basket_id",
            "target_item",
            "target_name",
            "recommendations",
        ]
    ]
)

print("Sample misses")
display(
    evaluation_df[evaluation_df["hit_at_k"].eq(0)]
    .head(20)[
        [
            "store_code",
            "basket_id",
            "target_item",
            "target_name",
            "recommendations",
        ]
    ]
)

Sample successful recommendations


,store_code,basket_id,target_item,target_name,recommendations
10,002,002|2026-07-01|8C20020035538,101024011706,MANGGIS,"[101011000528, 101024011706, 101006005937, 101..."
21,002,002|2026-07-01|8C20020035598,101003069991,MERUBALI COCONUT CHIPS CHOCOLATE 50GR,"[101003069991, 101024011706]"
42,002,002|2026-07-01|8C40020029010,101091080200,BCC TIRAMISU BERLINER,[101091080200]
71,002,002|2026-07-02|8C20020035654,101024011683,SUNPRIDE PISANG CAVENDISH,"[101003069991, 101024055997, 101003048125, 101..."
79,002,002|2026-07-02|8C20020035720,101003056140,MERUBALI COCONUT CHIPS 50GR,"[101003056140, 101024011706, 101003048125, 101..."
120,002,002|2026-07-02|8C40020029279,101091082653,BCC STRAWBERRY DONUT,"[101091073151, 101091082653, 101091073153]"
123,002,002|2026-07-02|8C40020029295,101091082653,BCC STRAWBERRY DONUT,"[101091073151, 101091082653, 101091073153]"
125,002,002|2026-07-02|8C40020029304,101067080289,RTE-P SALAD BUFFET PEPITO,"[101067080289, 101091073120, 101009000755]"
129,002,002|2026-07-02|8C50020025448,101091085457,BCC SOURDOUGH DINNER ROLL,[101091085457]
152,002,002|2026-07-03|8C20020036021,101024035322,JERUK WOGAN,"[101024026542, 101024035322, 101024032294, 101..."


Sample misses


,store_code,basket_id,target_item,target_name,recommendations
0,002,002|2026-07-01|8C20020035464,101012050374,BELCUBE PARTY CUBE PLAIN 78GR,[]
1,002,002|2026-07-01|8C20020035473,101006032292,BANGO MANIS REF 400ML,"[101024011683, 101024035322]"
2,002,002|2026-07-01|8C20020035480,101006002816,INDOFOOD SAOS TOMAT 140ML,"[101024011683, 101023011544, 101026074480, 101..."
3,002,002|2026-07-01|8C20020035495,101003005428,VERKADE GINGER 150GR,"[101023011607, 101003069991, 101003048125, 101..."
4,002,002|2026-07-01|8C20020035497,101006030211,BOTAN MACKEREL BSR 425 GR,[]
5,002,002|2026-07-01|8C20020035501,101023011599,KYURI/ TIMUN JEPANG KG,"[101024011707, 101024035322, 101024011706, 101..."
6,002,002|2026-07-01|8C20020035505,101044079599,PASEO TISSUE FACIAL ELG ULTRA SOFT 330S,[]
7,002,002|2026-07-01|8C20020035512,101023011618,CABE RAWIT HIJAU KG,"[101023011544, 101023011607, 101023011588, 101..."
8,002,002|2026-07-01|8C20020035520,101003047976,DORITOS COOL RANCH TORTILLA CHIPS 7 OZ,"[101006006633, 101003069991, 101006003863, 101..."
9,002,002|2026-07-01|8C20020035524,101078076145,SM TAPE ULI,[]
